In [ ]:
import sys
import os
from pathlib import Path

# Add project root to path (adjust if notebook is in a subfolder)
project_root = Path.cwd().parent  # if notebook is in experiments/ or similar
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


import warnings

# Suppress the specific future warning from torchrl
warnings.filterwarnings(
    "ignore", 
    category=FutureWarning, 
    module="torchrl.modules.mcts.scores"
)

import torch
from torchrl.envs import EnvBase
from torchrl.data import (
    Composite, 
    Unbounded, 
    Bounded,
    Stacked
    
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

import matplotlib.pyplot as plt
import numpy as np

from urbanmarl.envs.urbanmarl_env import UrbanMARLEnv
from urbanmarl.models.urban_map import VectorizedUrbanMap

In [ ]:
from tensordict import (
    is_tensor_collection,
    LazyStackedTensorDict,
    TensorDictBase,
    unravel_key,
)

In [ ]:
batch_size = 2
volume_size = (500, 500, 200)

env = VectorizedUrbanMap(
    batch_size=batch_size,
    volume_size=volume_size,
    device=device)

# Initialize internal environmental generation seeds dynamically
# B = torch.Size([batch_size])
# alpha = torch.rand(B, device=device) * 0.7 + 0.1
# beta = torch.randint(100, 750, B, device=device).float()
# gamma = torch.rand(B, device=device) * 42.0 + 8.0
# map_engine.generate_batch_maps(alpha, beta, gamma)
env.reset()

In [ ]:
env.plot(batch_idx = 0)

## Test Position Generation

In [ ]:
# test large number of outdoor positions

num_pos=10000
pos = env.gen_pos(num_pos=num_pos, min_z=1.5, max_z=1.5, outdoor=True)
count = 0
for b in range(env.batch_size):
    for p in pos[b]:
        x, y, z = p
        x = int(((x - env.x_min) ).round())
        y = int(((y - env.y_min) ).round())
        v = env.height_maps[b][ x, y]
        if (env.height_maps[b][ x, y] != 0):
            count += 1
        assert (env.height_maps[b][ x, y] == 0).all(), f"Position {b, p, x, y, v} contains non-zero elements!"
        # print(x, y, env.height_maps[b][ x, y])
print(f"fail: {count} {count/num_pos}")

## Test position generation

In [ ]:
# Normalized
num_uavs = 2
num_ues = 3
min_z = 1.0
max_z = 2.0
uav_pos = env.gen_pos(num_pos=num_uavs, min_z=1.5, max_z=150, outdoor=True, normalized=True)
print(f"Positions shape: {uav_pos.shape}")
print(f"Positions: \n{uav_pos}")
ue_pos = env.gen_pos(num_pos=num_ues, min_z=1.5, max_z=1.5, outdoor=True, normalized=True)
print(f"Positions shape: {ue_pos.shape}")
print(f"Positions: \n{ue_pos}")

In [ ]:
# Not normalized
num_uavs = 2
num_ues = 3
min_z = 1.0
max_z = 2.0
uav_pos = env.gen_pos(num_pos=num_uavs, min_z=1.5, max_z=150, outdoor=True)
print(f"Positions shape: {uav_pos.shape}, device: {uav_pos.device}")
print(f"Positions: \n{uav_pos}")
ue_pos = env.gen_pos(num_pos=num_ues, min_z=1.5, max_z=1.5, outdoor=True)
print(f"Positions shape: {ue_pos.shape}, device: {ue_pos.device}")
print(f"Positions: \n{ue_pos}")


In [ ]:
uav_in_grid = env.pos_to_grid(uav_pos)
uav_in_grid

## Test LoS

In [ ]:
print(uav_pos)
print(ue_pos)

In [ ]:
env.check_los_batch(uav_pos, ue_pos)

In [ ]:
env.check_los_batch(uav_pos, uav_pos)

In [ ]:
uav_g = env.pos_to_grid(uav_pos)
ue_g = env.pos_to_grid(ue_pos)


In [ ]:
urban = 1
uav = 0
ue = 2
print(uav_pos[urban,uav])
print(uav_g[urban,uav])
print(ue_pos[urban,ue])
print(ue_g[urban,ue])
h_map = env.height_maps[urban].cpu().numpy()

h_map[*uav_g[urban,uav].cpu().numpy()] = uav_pos[urban,uav,2].cpu().numpy()
h_map[*ue_g[urban,uav].cpu().numpy()] = ue_pos[urban,ue,2].cpu().numpy()
plt.gca()
plt.imshow(h_map, origin='lower', cmap="viridis")
# plt.imshow(image, cmap='gray')      # Use cmap='gray' for grayscale


plt.colorbar()                     # Optional: adds a scale
plt.show()

In [ ]:
# Reset agent position
def _reset_positions(self) -> TensorDictBase:
    self.agent_pos = self.get

In [ ]:
num_pos = 3
min_z = 1.0
max_z = 2.0

xy_indices = (env.height_maps==0).nonzero()
# xy_indices = torch.where(env.height_maps[b]==0)
# Generate a random permutation of the length of your indices
shuffled_positions = torch.randperm(len(xy_indices))
# Slice the first N positions
selected_positions = shuffled_positions[:num_pos]
# Extract the actual indices
sampled_indices = xy_indices[selected_positions]
# z_column
z_column = min_z + torch.rand(size=(num_pos, 1), device=env.device) * (max_z - min_z)
# pos = torch.cat([xy_indices[:, 0], xy_indices[:, 1], z_column[:,0]], dim=0)
pos = torch.cat([sampled_indices,  z_column], dim=1)

print(pos)
    

In [ ]:
print(env.device)
for attr, value in env.__dict__.items():
    print(attr, value)

In [ ]:
images = env.height_maps

In [ ]:
images[0].cpu().numpy()

In [ ]:
plt.imshow(images[0].cpu().numpy(), cmap='gray')      # Use cmap='gray' for grayscale
plt.colorbar()                     # Optional: adds a scale
plt.show()

In [ ]:
data = np.random.random((100, 100)) # Random 2D array
plt.imshow(data, cmap='gray')      # Use cmap='gray' for grayscale
plt.colorbar()                     # Optional: adds a scale
plt.show()